# 3 - MLP Put Option Pricer

The same architecture and training schedule as the call pricer, fitted to out-of-the-money puts (`underlying_value < strike_price`).

Unlike the call model, the put model does improve through the lr=1e-6 fine-tuning stage before degrading, reaching the best overall result of either model.

In [1]:
# @title Default title text
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, LeakyReLU, BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split


In [2]:
# Hyperparams
n_units = 400
layers = 4
n_batch = 4096
n_epochs = 200

In [3]:
data= pd.read_excel('../data/ASIANPAINT_Dataset.xlsx')
estimate_σ = lambda arr: (np.diff(arr) / arr[:-1]).std()
data['sigma_20'] = data.close.rolling(20).apply(estimate_σ)
data.dropna(subset=['sigma_20'], inplace=True)

data.head()

,Date,Expiry,t,strike_price,underlying_value,sigma,r,close,sigma_20
19,2020-01-01,2020-01-30,29,1660,1793.2,0.008151,0.0494,104.90,23.727248
20,2020-01-01,2020-01-30,29,1680,1793.2,0.008151,0.0494,136.00,1.405593
21,2020-01-01,2020-01-30,29,1560,1793.2,0.008151,0.0494,252.00,1.351294
22,2020-01-01,2020-01-30,29,1580,1793.2,0.008151,0.0494,279.15,1.012983
23,2020-01-01,2020-01-30,29,1600,1793.2,0.008151,0.0494,225.05,1.011580


In [4]:

data = data[data.underlying_value < data.strike_price]
data.head(30)

,Date,Expiry,t,strike_price,underlying_value,sigma,r,close,sigma_20
31,2020-01-01,2020-01-30,29,2000,1793.20,0.008151,0.0494,2.00,0.631502
32,2020-01-01,2020-02-27,57,2000,1793.20,0.008151,0.0494,16.25,1.713754
33,2020-01-01,2020-02-27,57,2020,1793.20,0.008151,0.0494,13.95,1.711956
34,2020-01-01,2020-02-27,57,1940,1793.20,0.008151,0.0494,17.75,1.712428
35,2020-01-01,2020-02-27,57,1960,1793.20,0.008151,0.0494,14.95,1.703450
36,2020-01-01,2020-02-27,57,1980,1793.20,0.008151,0.0494,18.85,1.673592
37,2020-01-01,2020-02-27,57,1900,1793.20,0.008151,0.0494,33.10,1.667026
38,2020-01-01,2020-02-27,57,1920,1793.20,0.008151,0.0494,22.50,1.664862
39,2020-01-01,2020-02-27,57,1840,1793.20,0.008151,0.0494,42.90,1.667627
40,2020-01-01,2020-02-27,57,1860,1793.20,0.008151,0.0494,42.85,1.668576


In [5]:

put_X_train, put_X_test, put_y_train, put_y_test = train_test_split(data.drop(['close','Date','Expiry','sigma'], axis=1),
                                                                    (data.close),
                                                                      test_size=0.2, random_state=42)

In [7]:
model = Sequential()
model.add(BatchNormalization(input_shape=(put_X_train.shape[1],)))
model.add(Dense(n_units, input_dim=put_X_train.shape[1]))
model.add(LeakyReLU())

for _ in range(layers - 1):
    model.add(Dense(n_units))
    model.add(BatchNormalization())
    model.add(LeakyReLU())

model.add(Dense(1, activation='relu'))

model.compile(loss='mse', optimizer=Adam(lr=1e-5))

In [8]:
model.summary()


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 batch_normalization (Batch  (None, 5)                 20        
 Normalization)                                                  
                                                                 
 dense (Dense)               (None, 400)               2400      
                                                                 
 leaky_re_lu (LeakyReLU)     (None, 400)               0         
                                                                 
 dense_1 (Dense)             (None, 400)               160400    
                                                                 
 batch_normalization_1 (Bat  (None, 400)               1600      
 chNormalization)                                                
                                                                 
 leaky_re_lu_1 (LeakyReLU)   (None, 400)              

In [9]:
history = model.fit(put_X_train, put_y_train,
                    batch_size=n_batch, epochs=n_epochs,
                    validation_split = 0.01,
                    callbacks=[TensorBoard()],
                    verbose=1)

Epoch 1/200
3/3 [==============================] - 4s 390ms/step - loss: 8201.8418 - val_loss: 8061.9775
Epoch 2/200
3/3 [==============================] - 1s 258ms/step - loss: 7431.6304 - val_loss: 8061.9775
Epoch 3/200
3/3 [==============================] - 1s 255ms/step - loss: 6897.9263 - val_loss: 8061.9775
Epoch 4/200
3/3 [==============================] - 1s 252ms/step - loss: 6602.5742 - val_loss: 8061.9775
Epoch 5/200
3/3 [==============================] - 1s 253ms/step - loss: 6419.9082 - val_loss: 8061.9775
Epoch 6/200
3/3 [==============================] - 1s 360ms/step - loss: 6293.7529 - val_loss: 8061.9775
Epoch 7/200
3/3 [==============================] - 1s 431ms/step - loss: 6180.9331 - val_loss: 8061.9775
Epoch 8/200
3/3 [==============================] - 1s 415ms/step - loss: 6073.5122 - val_loss: 8061.9775
Epoch 9/200
3/3 [==============================] - 1s 311ms/step - loss: 5971.6548 - val_loss: 8061.9775
Epoch 10/200
3/3 [==============================] - 1s 

In [10]:
model.save('mlp1-put10.h5')


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [13]:
put_y_pred = model.predict(put_X_test)


88/88 [==============================] - 0s 3ms/step


In [14]:
#import sklearn.metrics
from sklearn.metrics import mean_squared_error,mean_absolute_error
accuracy = mean_squared_error(put_y_pred,put_y_test)
print(accuracy)
accuracy1=mean_absolute_error(put_y_pred,put_y_test)
print(accuracy1)
print(np.sqrt(accuracy))

1729.242075275774
28.834030768070207
41.58415654159374


In [15]:
model.compile(loss='mse', optimizer=Adam(lr=1e-6))


In [16]:
history = model.fit(put_X_train, put_y_train,
                    batch_size=n_batch, epochs=20,
                    validation_split = 0.01,
                    callbacks=[TensorBoard()],
                    verbose=1)

Epoch 1/20
3/3 [==============================] - 3s 400ms/step - loss: 1956.8217 - val_loss: 32688.6191
Epoch 2/20
3/3 [==============================] - 1s 262ms/step - loss: 1789.4172 - val_loss: 43921.6914
Epoch 3/20
3/3 [==============================] - 1s 330ms/step - loss: 1560.0287 - val_loss: 39754.9766
Epoch 4/20
3/3 [==============================] - 1s 265ms/step - loss: 1486.8787 - val_loss: 40729.9102
Epoch 5/20
3/3 [==============================] - 1s 270ms/step - loss: 1387.0897 - val_loss: 39336.7344
Epoch 6/20
3/3 [==============================] - 1s 253ms/step - loss: 1222.8671 - val_loss: 33009.6016
Epoch 7/20
3/3 [==============================] - 1s 385ms/step - loss: 1212.5889 - val_loss: 25030.1641
Epoch 8/20
3/3 [==============================] - 1s 415ms/step - loss: 1146.1025 - val_loss: 15661.9551
Epoch 9/20
3/3 [==============================] - 1s 415ms/step - loss: 1108.5862 - val_loss: 8453.9180
Epoch 10/20
3/3 [==============================] - 1s 33

In [17]:
model.save('mlp1-20.h5')


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [18]:
put_y_pred = model.predict(put_X_test)


88/88 [==============================] - 0s 3ms/step


In [19]:
#import sklearn.metrics
from sklearn.metrics import mean_squared_error,mean_absolute_error
accuracy = mean_squared_error(put_y_pred,put_y_test)
print(accuracy)
accuracy1=mean_absolute_error(put_y_pred,put_y_test)
print(accuracy1)
print(np.sqrt(accuracy))

1001.1096671346927
20.88693884297913
31.640317114951497


In [20]:
model.compile(loss='mse', optimizer=Adam(lr=1e-7))


In [21]:
history = model.fit(put_X_train, put_y_train,
                    batch_size=n_batch, epochs=10,
                    validation_split = 0.01,
                    callbacks=[TensorBoard()],
                    verbose=1)

Epoch 1/10
3/3 [==============================] - 3s 357ms/step - loss: 1631.6332 - val_loss: 3359.9661
Epoch 2/10
3/3 [==============================] - 1s 259ms/step - loss: 1424.8508 - val_loss: 2773.3313
Epoch 3/10
3/3 [==============================] - 1s 271ms/step - loss: 1096.7952 - val_loss: 2438.2949
Epoch 4/10
3/3 [==============================] - 1s 438ms/step - loss: 1078.2729 - val_loss: 3223.4548
Epoch 5/10
3/3 [==============================] - 1s 449ms/step - loss: 1008.2402 - val_loss: 3224.2439
Epoch 6/10
3/3 [==============================] - 1s 379ms/step - loss: 976.8469 - val_loss: 3302.0420
Epoch 7/10
3/3 [==============================] - 1s 384ms/step - loss: 956.2967 - val_loss: 2648.1431
Epoch 8/10
3/3 [==============================] - 1s 269ms/step - loss: 899.7511 - val_loss: 1896.5032
Epoch 9/10
3/3 [==============================] - 1s 399ms/step - loss: 907.3470 - val_loss: 1681.7014
Epoch 10/10
3/3 [==============================] - 1s 266ms/step - l

In [24]:
model.save('mlp1-115.h5')
put_y_pred = model.predict(put_X_test)

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


88/88 [==============================] - 1s 5ms/step


In [25]:
#import sklearn.metrics
from sklearn.metrics import mean_squared_error,mean_absolute_error
accuracy = mean_squared_error(put_y_pred,put_y_test)
print(accuracy)
accuracy1=mean_absolute_error(put_y_pred,put_y_test)
print(accuracy1)
print(np.sqrt(accuracy))

1283.8525172093089
24.06249676173596
35.83088775357525


In [26]:
model.compile(loss='mse', optimizer=Adam(lr=1e-8))
history = model.fit(put_X_train, put_y_train,
                    batch_size=n_batch, epochs=5,
                    validation_split = 0.01,
                    callbacks=[TensorBoard()],
                    verbose=1)

Epoch 1/5
3/3 [==============================] - 3s 364ms/step - loss: 1121.4915 - val_loss: 2347.2339
Epoch 2/5
3/3 [==============================] - 1s 249ms/step - loss: 1085.4633 - val_loss: 3248.0176
Epoch 3/5
3/3 [==============================] - 1s 244ms/step - loss: 936.5619 - val_loss: 1815.8929
Epoch 4/5
3/3 [==============================] - 1s 271ms/step - loss: 926.8187 - val_loss: 1637.3362
Epoch 5/5
3/3 [==============================] - 1s 268ms/step - loss: 871.6321 - val_loss: 1674.3173


In [27]:
model.save('mlp1-120.h5')
put_y_pred = model.predict(put_X_test)

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


88/88 [==============================] - 0s 3ms/step


In [28]:
#import sklearn.metrics
from sklearn.metrics import mean_squared_error,mean_absolute_error
accuracy = mean_squared_error(put_y_pred,put_y_test)
print(accuracy)
accuracy1=mean_absolute_error(put_y_pred,put_y_test)
print(accuracy1)
print(np.sqrt(accuracy))

1176.6868988821611
23.285120192498447
34.30287012601367
